<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SeedVR2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/SeedVR2_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🚀 SeedVR2 — 1-Step Diffusion Video Super-Resolution & Restoration (`3B` & `7B`, 1080p / 4K)

**Models:** [`ByteDance-Seed/SeedVR2-3B`](https://huggingface.co/ByteDance-Seed/SeedVR2-3B) &nbsp;|&nbsp; [`ByteDance-Seed/SeedVR2-7B`](https://huggingface.co/ByteDance-Seed/SeedVR2-7B) &nbsp;|&nbsp; **FP8/FP16 Weights:** [`numz/SeedVR2_comfyUI`](https://huggingface.co/numz/SeedVR2_comfyUI) &nbsp;|&nbsp; **Standalone Engine:** [`numz/ComfyUI-SeedVR2_VideoUpscaler`](https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler) &nbsp;|&nbsp; **License:** Apache 2.0

**SeedVR2** is ByteDance Seed's **one-step diffusion video restoration and super-resolution** model built on a Shifted-Window Diffusion Transformer (`NaDiT`) and Causal Video VAE (`VideoAutoencoderKLWrapper`). Unlike traditional frame-by-frame upscalers (Real-ESRGAN) that flicker on hair, skin pores, and teeth, SeedVR2 restores temporally coherent **1080p, 1440p, and 4K** video in a **single diffusion step**.

### ✨ Key Features of This Colab Notebook
- **📂 Cross-Notebook Drive Video Picker (1-Click Avatar & AI Video Upscaling)**:
  - Automatically scans your Google Drive output folders (`/content/drive/MyDrive/AEI_3D_Out/` and `/content/drive/MyDrive/InfiniteTalk_Outputs/`) for recent `.mp4` videos from **`InfiniteTalk`**, **`LongCat-Video-Avatar-1.5`**, **`Wan2.2`**, **`Wan-VACE`**, **`LTX-Video-2.5`**, **`MiniMax-H3`**, **`FramePack`**, and **`LivePortrait`**.
  - Upscale 480p talking-avatar or AI videos to **1080p or 4K** while **preserving the original audio track** via lossless `ffmpeg` audio remuxing.
- **All Official 3B & 7B Precision Variants (`FP8` / `FP16` / `7B-Sharp`)**:
  - **`SeedVR2-3B FP8`** (`seedvr2_ema_3b_fp8_e4m3fn.safetensors`) — fast & lightweight (`3.4 GB`), runs on **Colab T4 (16 GB)** and **L4 (24 GB)**.
  - **`SeedVR2-3B FP16`** (`seedvr2_ema_3b_fp16.safetensors`) — full FP16 precision (`6.8 GB`) for **L4 (24 GB)** and **A100**.
  - **`SeedVR2-7B FP8 Mixed` & `SeedVR2-7B Sharp FP8 Mixed`** (`8.5 GB`) — 7B restoration & crisp texture recovery on **L4 (24 GB)** (with BlockSwap) and **A100**.
  - **`SeedVR2-7B FP16` & `SeedVR2-7B Sharp FP16`** (`16.5 GB`) — flagship 7B weights for **A100 (40/80 GB)**.
- **Single-GPU Memory Engineering (`BlockSwap` + Tiled Causal VAE + Streaming Chunks)**:
  - Dynamically swaps transformer blocks (`--blocks_to_swap 0..36`) between CPU RAM and GPU VRAM, tiles VAE encoding/decoding (`--vae_encode_tiled --vae_decode_tiled`), and processes long videos in memory-bounded streaming chunks (`--chunk_size`) with temporal overlap blending (`--temporal_overlap`) and perceptual `lab` / `wavelet` color correction.

| GPU Tier | VRAM | Recommended Model Variant | Recommended `blocks_to_swap` | Target Output |
| :--- | :--- | :--- | :--- | :--- |
| **Colab T4** | 16 GB | `SeedVR2-3B FP8` | `16`–`24` + Tiled VAE | `720p` / `1080p` (`batch_size=5`) |
| **Colab L4** | 24 GB | `SeedVR2-3B FP16` or `SeedVR2-7B Sharp FP8` | `0` (3B) / `12`–`16` (7B) + Tiled VAE | `1080p` / `1440p` (`batch_size=9`–`17`) |
| **Colab A100** | 40 / 80 GB | `SeedVR2-7B Sharp FP16` | `0` (Full GPU) + Tiled VAE for 4K | `1080p` / `4K (2160p)` (`batch_size=17`–`33`) |

In [ ]:
# @title STEP 1 — Clone SeedVR2 Standalone Engine & Install Dependencies
# @markdown Clones `numz/ComfyUI-SeedVR2_VideoUpscaler` (which includes the standalone `inference_cli.py` engine),
# @markdown installs required packages, and auto-detects optimal defaults for your Colab GPU.

import os, sys, subprocess, time, shutil, torch

if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU detected. Go to Runtime -> Change runtime type -> select T4, L4, or A100 GPU.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB VRAM | PyTorch {torch.__version__} (CUDA {torch.version.cuda})')

SEEDVR2_REPO_DIR = '/content/SeedVR2_VideoUpscaler'
if not os.path.exists(os.path.join(SEEDVR2_REPO_DIR, 'inference_cli.py')):
    print(f'[Git] Cloning https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git -> {SEEDVR2_REPO_DIR} ...')
    if os.path.exists(SEEDVR2_REPO_DIR):
        shutil.rmtree(SEEDVR2_REPO_DIR)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git',
        SEEDVR2_REPO_DIR,
    ], check=True)
else:
    print(f'[Git] SeedVR2 standalone repository already present at {SEEDVR2_REPO_DIR}')

if SEEDVR2_REPO_DIR not in sys.path:
    sys.path.insert(0, SEEDVR2_REPO_DIR)

t0 = time.time()
print('[pip] Installing SeedVR2 dependencies + gradio==5.49.1 + tqdm==4.66.5 ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'omegaconf>=2.3.0',
    'diffusers>=0.33.1',
    'peft>=0.17.0',
    'rotary_embedding_torch>=0.5.3',
    'einops',
    'gguf',
    'safetensors',
    'opencv-python-headless',
    'matplotlib',
    'psutil',
    'huggingface-hub>=0.27.0',
    'tqdm==4.66.5',
    'gradio==5.49.1',
], check=True)
print(f'[Install] Dependencies installed in {time.time() - t0:.1f}s.')

if vram_gb < 20.0:
    RECOMMENDED_VARIANT = 'SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)'
    RECOMMENDED_BLOCKS_SWAP = 16
    RECOMMENDED_BATCH = 5
elif vram_gb < 34.0:
    RECOMMENDED_VARIANT = 'SeedVR2-3B FP16 (Standard 3B, L4 24GB / A100)'
    RECOMMENDED_BLOCKS_SWAP = 0
    RECOMMENDED_BATCH = 9
else:
    RECOMMENDED_VARIANT = 'SeedVR2-7B Sharp FP16 (Flagship Sharp 7B, A100 40GB/80GB)'
    RECOMMENDED_BLOCKS_SWAP = 0
    RECOMMENDED_BATCH = 17

print(f'[Auto-Tune] Recommended Variant: {RECOMMENDED_VARIANT} | blocks_to_swap={RECOMMENDED_BLOCKS_SWAP} | batch_size={RECOMMENDED_BATCH}')

In [ ]:
# @title STEP 2 — Mount Google Drive & Cache SeedVR2 Weights (`3B` / `7B` + Causal VAE)
# @markdown Mounts Google Drive and caches the shared `ema_vae_fp16.safetensors` plus your chosen
# @markdown SeedVR2 3B/7B DiT checkpoint under `/content/drive/MyDrive/AEI_3D_Cache/SeedVR2/models`.

import os, sys, time, pathlib, shutil
from huggingface_hub import hf_hub_download
from tqdm.notebook import tqdm

# (hf_repo, filename, approx_gb, max_swap_blocks, description)
SEEDVR2_MODEL_REGISTRY = {
    'SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)': (
        'numz/SeedVR2_comfyUI',
        'seedvr2_ema_3b_fp8_e4m3fn.safetensors',
        3.4,
        32,
        '3B FP8 E4M3 — lowest VRAM footprint, ideal for T4 16GB and fast L4 1080p upscaling',
    ),
    'SeedVR2-3B FP16 (Standard 3B, L4 24GB / A100)': (
        'numz/SeedVR2_comfyUI',
        'seedvr2_ema_3b_fp16.safetensors',
        6.8,
        32,
        '3B Full FP16 — clean 1-step diffusion restoration on 24GB L4 and 40GB A100',
    ),
    'SeedVR2-7B FP8 Mixed (High Detail 7B, L4 24GB / A100)': (
        'AInVFX/SeedVR2_comfyUI',
        'seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors',
        8.5,
        36,
        '7B FP8 mixed-precision — fits 24GB L4 (with BlockSwap 12–16) and A100',
    ),
    'SeedVR2-7B Sharp FP8 Mixed (Extra Crisp 7B, L4 24GB / A100)': (
        'AInVFX/SeedVR2_comfyUI',
        'seedvr2_ema_7b_sharp_fp8_e4m3fn_mixed_block35_fp16.safetensors',
        8.5,
        36,
        '7B Sharp FP8 mixed-precision — enhanced micro-texture & facial detail on L4/A100',
    ),
    'SeedVR2-7B FP16 (Flagship 7B, A100 40GB/80GB)': (
        'numz/SeedVR2_comfyUI',
        'seedvr2_ema_7b_fp16.safetensors',
        16.5,
        36,
        '7B Full FP16 — flagship fidelity for A100 40GB/80GB (or L4 with BlockSwap 24)',
    ),
    'SeedVR2-7B Sharp FP16 (Flagship Sharp 7B, A100 40GB/80GB)': (
        'numz/SeedVR2_comfyUI',
        'seedvr2_ema_7b_sharp_fp16.safetensors',
        16.5,
        36,
        '7B Sharp Full FP16 — maximum sharpness for 1080p/4K avatar & video restoration',
    ),
}

VAE_REPO = 'numz/SeedVR2_comfyUI'
VAE_FILENAME = 'ema_vae_fp16.safetensors'

DEFAULT_DIT_CHOICE = 'SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)'  # @param ['SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)', 'SeedVR2-3B FP16 (Standard 3B, L4 24GB / A100)', 'SeedVR2-7B FP8 Mixed (High Detail 7B, L4 24GB / A100)', 'SeedVR2-7B Sharp FP8 Mixed (Extra Crisp 7B, L4 24GB / A100)', 'SeedVR2-7B FP16 (Flagship 7B, A100 40GB/80GB)', 'SeedVR2-7B Sharp FP16 (Flagship Sharp 7B, A100 40GB/80GB)']

try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/SeedVR2')
    HAS_DRIVE = True
except Exception as exc:
    print(f'[Cache] Google Drive not mounted ({exc}); using /content/hf_cache/SeedVR2.')
    CACHE_ROOT = pathlib.Path('/content/hf_cache/SeedVR2')
    HAS_DRIVE = False

MODEL_DIR = CACHE_ROOT / 'models'
HF_HOME_DIR = CACHE_ROOT / 'hf_home'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME_DIR / 'hub')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

OUT_DIR = pathlib.Path('/content/seedvr2_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if HAS_DRIVE:
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SeedVR2')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_OUT = OUT_DIR

def ensure_seedvr2_weights(variant_key: str) -> str:
    if variant_key not in SEEDVR2_MODEL_REGISTRY:
        raise ValueError(f'Unknown SeedVR2 variant: {variant_key}')
    vae_path = MODEL_DIR / VAE_FILENAME
    if not vae_path.exists() or vae_path.stat().st_size < 100_000_000:
        print(f'[Cache] Downloading shared Causal Video VAE ({VAE_FILENAME}) ...')
        hf_hub_download(
            repo_id=VAE_REPO,
            filename=VAE_FILENAME,
            local_dir=str(MODEL_DIR),
            local_dir_use_symlinks=False,
        )
    else:
        print(f'[Cache] Shared VAE already cached: {vae_path.name} ({vae_path.stat().st_size / (1024**3):.2f} GB)')

    repo_id, dit_filename, approx_gb, _, blurb = SEEDVR2_MODEL_REGISTRY[variant_key]
    dit_path = MODEL_DIR / dit_filename
    if not dit_path.exists() or dit_path.stat().st_size < 500_000_000:
        print(f'[Cache] Downloading {variant_key} ({dit_filename}, ~{approx_gb} GB) from {repo_id} ...')
        t0 = time.time()
        hf_hub_download(
            repo_id=repo_id,
            filename=dit_filename,
            local_dir=str(MODEL_DIR),
            local_dir_use_symlinks=False,
        )
        print(f'[Cache] Downloaded {dit_filename} in {time.time() - t0:.1f}s.')
    else:
        print(f'[Cache] DiT model already cached: {dit_path.name} ({dit_path.stat().st_size / (1024**3):.2f} GB)')
    return dit_filename

active_dit_file = ensure_seedvr2_weights(DEFAULT_DIT_CHOICE)
print(f'[Setup] Model directory : {MODEL_DIR}')
print(f'[Setup] Default DiT     : {active_dit_file}')
print(f'[Setup] Output directory: {DRIVE_OUT}')

In [ ]:
# @title STEP 3 — SeedVR2 Upscaling Engine, Audio Preservation & Cross-Notebook Video Discovery
# @markdown Defines:
# @markdown - `discover_cross_notebook_videos()`: scans Google Drive for recent `.mp4` outputs from `InfiniteTalk`,
# @markdown   `LongCat-Video-Avatar`, `Wan2.2`, `Wan-VACE`, `LTX-Video-2.5`, `MiniMax-H3`, `FramePack`, and `LivePortrait`.
# @markdown - `SeedVR2Engine`: invokes `inference_cli.py` with BlockSwap, Tiled VAE, Streaming Chunks, and `lab`/`wavelet`
# @markdown   color correction, then automatically remuxes the original audio track via `ffmpeg`.

import os, sys, time, uuid, shutil, pathlib, subprocess, traceback
from typing import Dict, List, Optional, Tuple
import cv2, torch, numpy as np
from PIL import Image
from IPython.display import display, FileLink

CROSS_NOTEBOOK_SEARCH_DIRS = [
    ('InfiniteTalk', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/InfiniteTalk')),
    ('InfiniteTalk', pathlib.Path('/content/drive/MyDrive/InfiniteTalk_Outputs')),
    ('InfiniteTalk', pathlib.Path('/content/drive/MyDrive/InfiniteTalk_Colab/outputs')),
    ('LongCat-Avatar', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/LongCat-Video-Avatar-1.5')),
    ('LongCat-Avatar', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/LongCat-Video-Avatar')),
    ('Wan2.2', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan2.2')),
    ('Wan2.2-Animate', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan2.2-Animate')),
    ('Wan2.2-S2V', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan2.2-S2V')),
    ('Wan-VACE', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan-VACE')),
    ('LTX-Video-2.5', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/LTX-2-5')),
    ('LTX-Video-2.5', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/LTX-Video-2.5')),
    ('MiniMax-H3', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/MiniMax-H3')),
    ('FramePack', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/FramePack')),
    ('LivePortrait', pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/LivePortrait')),
]

def probe_video_info(video_path: str) -> Dict[str, object]:
    info = {'width': 0, 'height': 0, 'fps': 0.0, 'frames': 0, 'duration_s': 0.0, 'has_audio': False}
    if not video_path or not os.path.exists(str(video_path)):
        return info
    cap = cv2.VideoCapture(str(video_path))
    if cap.isOpened():
        info['width'] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        info['height'] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        info['fps'] = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)
        info['frames'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if info['fps'] > 0:
            info['duration_s'] = round(info['frames'] / info['fps'], 2)
        cap.release()
    try:
        res = subprocess.run(
            ['ffprobe', '-v', 'error', '-select_streams', 'a:0', '-show_entries', 'stream=codec_type', '-of', 'csv=p=0', str(video_path)],
            capture_output=True, text=True, timeout=10,
        )
        info['has_audio'] = ('audio' in (res.stdout or '').lower())
    except Exception:
        info['has_audio'] = False
    return info

def discover_cross_notebook_videos(max_results: int = 60) -> Dict[str, str]:
    """Scans Google Drive output folders across AEI notebooks and returns {display_label: abs_path}."""
    found = []
    seen_paths = set()
    for tag, folder in CROSS_NOTEBOOK_SEARCH_DIRS:
        if not folder.exists():
            continue
        for mp4 in folder.rglob('*.mp4'):
            try:
                resolved = str(mp4.resolve())
                if resolved in seen_paths:
                    continue
                seen_paths.add(resolved)
                st = mp4.stat()
                if st.st_size < 10_000:
                    continue
                found.append((st.st_mtime, tag, mp4, st.st_size))
            except Exception:
                continue
    # Also scan any other subfolder under /content/drive/MyDrive/AEI_3D_Out
    aei_out_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
    if aei_out_root.exists():
        for mp4 in aei_out_root.rglob('*.mp4'):
            try:
                if 'SeedVR2' in mp4.parts:
                    continue
                resolved = str(mp4.resolve())
                if resolved in seen_paths:
                    continue
                seen_paths.add(resolved)
                st = mp4.stat()
                if st.st_size >= 10_000:
                    found.append((st.st_mtime, mp4.parent.name, mp4, st.st_size))
            except Exception:
                continue

    found.sort(key=lambda x: x[0], reverse=True)
    mapping = {}
    for mtime, tag, mp4_path, sz_bytes in found[:max_results]:
        mb = sz_bytes / (1024 ** 2)
        ts = time.strftime('%Y-%m-%d %H:%M', time.localtime(mtime))
        label = f'[{tag}] {mp4_path.name} ({mb:.1f} MB — {ts})'
        mapping[label] = str(mp4_path)
    return mapping

def remux_audio_from_source(upscaled_video: str, source_video: str, output_video: str) -> str:
    """Copies the video stream from upscaled_video and muxes the audio track from source_video if present."""
    src_info = probe_video_info(source_video)
    if not src_info.get('has_audio'):
        if str(upscaled_video) != str(output_video):
            shutil.copy2(upscaled_video, output_video)
        return str(output_video)
    tmp_mux = str(output_video) + '.mux_tmp.mp4'
    cmd = [
        'ffmpeg', '-y',
        '-i', str(upscaled_video),
        '-i', str(source_video),
        '-map', '0:v:0',
        '-map', '1:a:0?',
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-b:a', '192k',
        '-shortest',
        tmp_mux,
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0 and os.path.exists(tmp_mux) and os.path.getsize(tmp_mux) > 1024:
        shutil.move(tmp_mux, str(output_video))
        print(f'[AudioMux] Preserved original audio stream -> {output_video}')
    else:
        print(f'[AudioMux] Warning: audio remux skipped ({res.stderr[-200:] if res.stderr else "no audio"}); keeping video stream.')
        if str(upscaled_video) != str(output_video):
            shutil.copy2(upscaled_video, output_video)
    return str(output_video)

class SeedVR2Engine:
    """Standalone wrapper around SeedVR2 inference_cli.py supporting 3B/7B FP8/FP16, BlockSwap, Tiled VAE & Audio Remux."""
    def __init__(self, repo_dir: str = SEEDVR2_REPO_DIR, model_dir: pathlib.Path = MODEL_DIR):
        self.repo_dir = pathlib.Path(repo_dir)
        self.cli_script = self.repo_dir / 'inference_cli.py'
        self.model_dir = pathlib.Path(model_dir)

    def upscale_media(
        self,
        input_path: str,
        variant_key: str = DEFAULT_DIT_CHOICE,
        resolution: int = 1080,
        max_resolution: int = 1920,
        batch_size: int = 5,
        uniform_batch_size: bool = True,
        temporal_overlap: int = 1,
        prepend_frames: int = 0,
        chunk_size: int = 0,
        color_correction: str = 'lab',
        blocks_to_swap: int = 0,
        swap_io_components: bool = False,
        vae_encode_tiled: bool = True,
        vae_decode_tiled: bool = True,
        vae_tile_size: int = 1024,
        vae_tile_overlap: int = 128,
        attention_mode: str = 'sdpa',
        preserve_audio: bool = True,
        use_10bit: bool = False,
        load_cap: int = 0,
        seed: int = 42,
    ) -> str:
        if not input_path or not os.path.exists(str(input_path)):
            raise FileNotFoundError(f'Input file not found: {input_path}')

        dit_filename = ensure_seedvr2_weights(variant_key)
        _, _, _, max_blocks, _ = SEEDVR2_MODEL_REGISTRY[variant_key]
        blocks_to_swap = max(0, min(int(blocks_to_swap), max_blocks))

        # Enforce 4n + 1 rule for batch_size
        b = max(1, int(batch_size))
        batch_size_4n1 = ((b - 1) // 4) * 4 + 1

        ext = pathlib.Path(str(input_path)).suffix.lower()
        is_image = ext in ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff')
        out_ext = '.png' if is_image else '.mp4'
        raw_out = OUT_DIR / f'seedvr2_raw_{int(time.time())}_{uuid.uuid4().hex[:6]}{out_ext}'

        offload_dev = 'cpu' if (blocks_to_swap > 0 or swap_io_components or vram_gb < 32.0) else 'none'
        cmd = [
            sys.executable, str(self.cli_script),
            str(input_path),
            '--output', str(raw_out),
            '--output_format', 'png' if is_image else 'mp4',
            '--model_dir', str(self.model_dir),
            '--dit_model', dit_filename,
            '--resolution', str(int(resolution)),
            '--max_resolution', str(int(max_resolution)),
            '--batch_size', str(1 if is_image else batch_size_4n1),
            '--color_correction', str(color_correction),
            '--attention_mode', str(attention_mode),
            '--dit_offload_device', offload_dev,
            '--vae_offload_device', 'cpu' if vram_gb < 32.0 else 'none',
            '--tensor_offload_device', 'cpu',
            '--blocks_to_swap', str(blocks_to_swap),
            '--seed', str(int(seed)),
            '--video_backend', 'ffmpeg',
        ]
        if not is_image and uniform_batch_size:
            cmd.append('--uniform_batch_size')
        if not is_image and int(temporal_overlap) > 0:
            cmd.extend(['--temporal_overlap', str(int(temporal_overlap))])
        if not is_image and int(prepend_frames) > 0:
            cmd.extend(['--prepend_frames', str(int(prepend_frames))])
        if not is_image and int(chunk_size) > 0:
            cmd.extend(['--chunk_size', str(int(chunk_size)), '--cache_dit', '--cache_vae'])
        if not is_image and int(load_cap) > 0:
            cmd.extend(['--load_cap', str(int(load_cap))])
        if swap_io_components and offload_dev != 'none':
            cmd.append('--swap_io_components')
        if vae_encode_tiled:
            cmd.extend(['--vae_encode_tiled', '--vae_encode_tile_size', str(int(vae_tile_size)), '--vae_encode_tile_overlap', str(int(vae_tile_overlap))])
        if vae_decode_tiled:
            cmd.extend(['--vae_decode_tiled', '--vae_decode_tile_size', str(int(vae_tile_size)), '--vae_decode_tile_overlap', str(int(vae_tile_overlap))])
        if use_10bit and not is_image:
            cmd.append('--10bit')

        print('[SeedVR2] Running command:\n  ' + ' '.join(cmd))
        t0 = time.time()
        proc = subprocess.Popen(
            cmd, cwd=str(self.repo_dir), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        log_lines = []
        for line in proc.stdout:
            line_s = line.rstrip()
            log_lines.append(line_s)
            print(line_s)
        proc.wait()
        if proc.returncode != 0 or not raw_out.exists():
            tail = '\n'.join(log_lines[-25:])
            raise RuntimeError(f'SeedVR2 CLI exited with code {proc.returncode}.\nLast output:\n{tail}')

        stem = pathlib.Path(str(input_path)).stem[:32]
        final_out = DRIVE_OUT / f'{stem}_{int(resolution)}p_{int(time.time())}{out_ext}'
        if not is_image and preserve_audio:
            remux_audio_from_source(str(raw_out), str(input_path), str(final_out))
        else:
            shutil.copy2(str(raw_out), str(final_out))
        print(f'[SeedVR2] Completed in {time.time() - t0:.1f}s -> {final_out}')
        return str(final_out)

ENGINE = SeedVR2Engine()
DISCOVERED_VIDEOS = discover_cross_notebook_videos()
print(f'[Core] SeedVR2Engine ready. Discovered {len(DISCOVERED_VIDEOS)} cross-notebook video(s) on Google Drive.')

In [ ]:
# @title STEP 4 — Launch Gradio UI (Cross-Notebook Video Upscaler & 4K Image Restorer)
# @markdown Launches the Gradio interface with the **📂 Cross-Notebook Drive Video Picker**,
# @markdown **1080p/4K Video Super-Resolution** (with audio preservation), and **Single-Image 4K Restoration**.

import os, sys, time, shutil, pathlib, traceback
import torch, gradio as gr
from IPython.display import clear_output, display, FileLink

RES_PRESET_MAP = {
    '720p HD (Short-side 720, Max 1280 — Fast T4/L4)': (720, 1280),
    '1080p Full HD (Short-side 1080, Max 1920 — Recommended)': (1080, 1920),
    '1440p QHD (Short-side 1440, Max 2560 — High Detail)': (1440, 2560),
    '2160p 4K UHD (Short-side 2160, Max 3840 — A100 / Tiled VAE)': (2160, 3840),
    'Custom (Use Short-Side & Max Resolution Sliders)': (0, 0),
}

def refresh_drive_picker():
    global DISCOVERED_VIDEOS
    DISCOVERED_VIDEOS = discover_cross_notebook_videos()
    choices = ['(None — Use Uploaded Video Below)'] + list(DISCOVERED_VIDEOS.keys())
    return gr.update(choices=choices, value=choices[0]), f'🔍 Found **{len(DISCOVERED_VIDEOS)}** video(s) across Drive output folders.'

def on_picker_select(selected_label: str):
    if not selected_label or selected_label.startswith('(None'):
        return None, 'Upload a video or pick a cross-notebook output from the dropdown.'
    path = DISCOVERED_VIDEOS.get(selected_label)
    if not path or not os.path.exists(path):
        return None, f'⚠️ File not found on disk: `{path}`'
    meta = probe_video_info(path)
    audio_badge = '🔊 Audio track detected (will be preserved)' if meta['has_audio'] else '🔇 No audio stream'
    summary = (
        f"📂 **Selected:** `{path}`\n"
        f"📐 **Source Resolution:** `{meta['width']} × {meta['height']}` @ `{meta['fps']:.1f} FPS` "
        f"(`{meta['frames']}` frames, `{meta['duration_s']}s`) | {audio_badge}"
    )
    return path, summary

def ui_upscale_video(
    picker_label, uploaded_video, variant_key, res_preset, custom_res, custom_max_res,
    batch_size, uniform_batch, temporal_overlap, prepend_frames, chunk_size, load_cap,
    color_correction, blocks_to_swap, swap_io, vae_encode_tiled, vae_decode_tiled, tile_size,
    preserve_audio, use_10bit, seed,
    progress=gr.Progress(),
):
    src_path = uploaded_video
    if (not src_path) and picker_label and not picker_label.startswith('(None'):
        src_path = DISCOVERED_VIDEOS.get(picker_label)
    if not src_path or not os.path.exists(str(src_path)):
        return None, '❌ Please upload a video or select one from the Cross-Notebook Drive Video Picker.'

    short_res, max_res = RES_PRESET_MAP.get(res_preset, (1080, 1920))
    if short_res == 0:
        short_res, max_res = int(custom_res), int(custom_max_res)

    try:
        src_meta = probe_video_info(src_path)
        progress(0.05, desc=f'Ensuring weights for {variant_key} ...')
        t0 = time.time()
        progress(0.12, desc=f"Upscaling {src_meta['width']}x{src_meta['height']} -> {short_res}p via {variant_key} ...")
        out_path = ENGINE.upscale_media(
            input_path=src_path,
            variant_key=variant_key,
            resolution=short_res,
            max_resolution=max_res,
            batch_size=int(batch_size),
            uniform_batch_size=bool(uniform_batch),
            temporal_overlap=int(temporal_overlap),
            prepend_frames=int(prepend_frames),
            chunk_size=int(chunk_size),
            color_correction=color_correction,
            blocks_to_swap=int(blocks_to_swap),
            swap_io_components=bool(swap_io),
            vae_encode_tiled=bool(vae_encode_tiled),
            vae_decode_tiled=bool(vae_decode_tiled),
            vae_tile_size=int(tile_size),
            preserve_audio=bool(preserve_audio),
            use_10bit=bool(use_10bit),
            load_cap=int(load_cap),
            seed=int(seed),
        )
        out_meta = probe_video_info(out_path)
        progress(1.0, desc='Done!')
        msg = (
            f"✅ **Upscaled in {time.time() - t0:.1f}s** (`{src_meta['width']}×{src_meta['height']}` → `{out_meta['width']}×{out_meta['height']}`)\n"
            f"💾 **Saved to Drive:** `{out_path}` (Audio preserved: `{out_meta['has_audio']}`)"
        )
        return out_path, msg
    except Exception as exc:
        traceback.print_exc()
        return None, f'❌ Error during SeedVR2 upscaling: {exc}'

def ui_upscale_image(
    img_path, variant_key, target_short_side, max_side, color_correction, blocks_to_swap, vae_tiled, tile_size, seed,
    progress=gr.Progress(),
):
    if not img_path or not os.path.exists(str(img_path)):
        return None, '❌ Please upload an image first.'
    try:
        progress(0.10, desc=f'Restoring & upscaling image to {int(target_short_side)}p ...')
        t0 = time.time()
        out_png = ENGINE.upscale_media(
            input_path=img_path,
            variant_key=variant_key,
            resolution=int(target_short_side),
            max_resolution=int(max_side),
            batch_size=1,
            color_correction=color_correction,
            blocks_to_swap=int(blocks_to_swap),
            vae_encode_tiled=bool(vae_tiled),
            vae_decode_tiled=bool(vae_tiled),
            vae_tile_size=int(tile_size),
            preserve_audio=False,
            seed=int(seed),
        )
        progress(1.0, desc='Done!')
        return out_png, f'✅ **Image restored in {time.time() - t0:.1f}s** → `{out_png}`'
    except Exception as exc:
        traceback.print_exc()
        return None, f'❌ Error: {exc}'

picker_initial_choices = ['(None — Use Uploaded Video Below)'] + list(DISCOVERED_VIDEOS.keys())

with gr.Blocks(title='SeedVR2 1080p/4K Video Upscaler', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# 🚀 SeedVR2 — 1-Step Diffusion Video Super-Resolution & Restoration (1080p / 4K)\n'
        'Upscale talking-avatar outputs (`InfiniteTalk`, `LongCat-Avatar`) and AI videos (`Wan2.2`, `Wan-VACE`, `LTX-2.5`, `FramePack`) '
        'to **1080p / 4K** in a single diffusion step while preserving original audio.'
    )

    with gr.Row():
        variant_dd = gr.Dropdown(
            choices=list(SEEDVR2_MODEL_REGISTRY.keys()),
            value=RECOMMENDED_VARIANT if 'RECOMMENDED_VARIANT' in globals() else DEFAULT_DIT_CHOICE,
            label='SeedVR2 DiT Model Variant (3B / 7B)',
            info='3B FP8 fits T4 (16 GB) & L4 (24 GB). 3B FP16 & 7B Sharp FP8 fit L4 (24 GB). 7B Sharp FP16 is ideal on A100 (40/80 GB).',
        )
        color_dd = gr.Dropdown(
            choices=['lab', 'wavelet', 'wavelet_adaptive', 'hsv', 'adain', 'none'],
            value='lab',
            label='Color Correction Method',
            info="'lab' (perceptual Lab color matching, recommended) or 'wavelet' preserves original skin tones and lighting accurately.",
        )
        blocks_swap_sl = gr.Slider(
            0, 36, value=RECOMMENDED_BLOCKS_SWAP if 'RECOMMENDED_BLOCKS_SWAP' in globals() else 12, step=2,
            label='BlockSwap (Transformer Blocks to Swap to CPU)',
            info='0 = keep full DiT in VRAM (fastest). Set 16–24 on T4 16GB or when running 7B on L4 24GB to prevent CUDA OOM.',
        )

    with gr.Tabs():
        with gr.Tab('1. 🎬 Video Super-Resolution & Cross-Notebook Avatar Upscaler'):
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Group():
                        gr.Markdown('### 📂 Cross-Notebook Drive Video Picker')
                        with gr.Row():
                            picker_dd = gr.Dropdown(
                                choices=picker_initial_choices,
                                value=picker_initial_choices[0],
                                label='Recent Outputs from InfiniteTalk / LongCat-Avatar / Wan2.2 / Wan-VACE / LTX-2.5 / MiniMax-H3',
                                info='Automatically discovers .mp4 files generated by other AEI Colab notebooks on your Google Drive.',
                                scale=4,
                            )
                            refresh_btn = gr.Button('🔄 Scan Drive Outputs', scale=1)
                        picker_info_md = gr.Markdown('Select a Drive video above or upload any `.mp4` below.')

                    video_in = gr.Video(
                        label='Input Video (.mp4 / .mov) — populated automatically when you pick a Drive video above',
                        height=280,
                    )
                    picker_dd.change(on_picker_select, inputs=[picker_dd], outputs=[video_in, picker_info_md])
                    refresh_btn.click(refresh_drive_picker, outputs=[picker_dd, picker_info_md])

                    res_preset_dd = gr.Dropdown(
                        choices=list(RES_PRESET_MAP.keys()),
                        value='1080p Full HD (Short-side 1080, Max 1920 — Recommended)',
                        label='Target Resolution Preset',
                        info='Sets the target short-side resolution (e.g., 1080 turns 480x832 portrait into 1080x1872, or 832x480 landscape into 1920x1080).',
                    )
                    with gr.Row():
                        batch_sl = gr.Slider(
                            1, 33, value=RECOMMENDED_BATCH if 'RECOMMENDED_BATCH' in globals() else 5, step=4,
                            label='Temporal Batch Size (4n + 1 frames)',
                            info='Must follow 4n+1 (1, 5, 9, 13, 17, 21...). Use 5 on T4 16GB, 9–13 on L4 24GB, 17–25 on A100.',
                        )
                        overlap_sl = gr.Slider(
                            0, 8, value=1, step=1, label='Temporal Overlap Frames',
                            info='Blends overlapping frames between temporal batches for flicker-free continuity.',
                        )
                        preserve_audio_cb = gr.Checkbox(
                            value=True, label='🔊 Preserve Original Audio Track (FFmpeg Remux)',
                            info='Automatically muxes the original speech/music audio stream from the source video into the upscaled MP4.',
                        )

                    with gr.Accordion('⚙️ Advanced Memory, Streaming & Tiled VAE Options', open=False):
                        with gr.Row():
                            custom_res_sl = gr.Slider(480, 2160, value=1080, step=16, label='Custom Short-Side Resolution (px)', info='Used when Target Resolution Preset = Custom.')
                            custom_max_sl = gr.Slider(0, 4096, value=1920, step=64, label='Custom Max Long-Side Resolution (0 = unlimited)', info='Caps the longest edge so ultra-wide videos stay within VRAM.')
                        with gr.Row():
                            chunk_sl = gr.Slider(0, 120, value=45, step=15, label='Streaming Chunk Size (frames, 0 = all at once)', info='Processes long avatar videos in memory-bounded chunks while keeping DiT/VAE cached.')
                            load_cap_sl = gr.Slider(0, 600, value=0, step=10, label='Max Frames Cap (0 = full video)', info='Set >0 to test upscaling on only the first N frames.')
                            prepend_sl = gr.Slider(0, 8, value=0, step=1, label='Prepend Reversed Frames', info='Prepends N reversed frames (auto-removed) if the very first frame has startup artifacts.')
                        with gr.Row():
                            vae_enc_cb = gr.Checkbox(value=True, label='Enable Tiled VAE Encoding', info='Splits high-res frames into overlapping tiles during VAE encoding to prevent VRAM spikes.')
                            vae_dec_cb = gr.Checkbox(value=True, label='Enable Tiled VAE Decoding', info='Splits high-res latents into overlapping tiles during VAE decoding (essential for 1080p/4K).')
                            tile_sz_sl = gr.Slider(512, 1536, value=1024, step=128, label='VAE Tile Size (px)', info='Tile size in pixels when Tiled VAE is enabled (default 1024).')
                        with gr.Row():
                            uniform_cb = gr.Checkbox(value=True, label='Uniform Batch Size Padding', info='Pads the final temporal batch to match batch_size for consistent quality.')
                            swap_io_cb = gr.Checkbox(value=False, label='Swap DiT I/O Layers', info='Offloads input/output patch embedding layers for extra VRAM savings on 16 GB GPUs.')
                            bit10_cb = gr.Checkbox(value=False, label='10-Bit x265 Encoding (--10bit)', info='Encodes 10-bit yuv420p10le H.265 video to eliminate gradient banding.')
                            seed_sl = gr.Slider(0, 2147483647, value=42, step=1, label='Seed', info='Deterministic random seed for diffusion restoration.')

                    upscale_vid_btn = gr.Button('🚀 Upscale Video to 1080p / 4K', variant='primary')
                with gr.Column(scale=2):
                    video_out = gr.Video(label='Upscaled Video Output (with Audio)', height=420)
                    vid_status_md = gr.Markdown('Ready.')

            upscale_vid_btn.click(
                ui_upscale_video,
                inputs=[
                    picker_dd, video_in, variant_dd, res_preset_dd, custom_res_sl, custom_max_sl,
                    batch_sl, uniform_cb, overlap_sl, prepend_sl, chunk_sl, load_cap_sl,
                    color_dd, blocks_swap_sl, swap_io_cb, vae_enc_cb, vae_dec_cb, tile_sz_sl,
                    preserve_audio_cb, bit10_cb, seed_sl,
                ],
                outputs=[video_out, vid_status_md],
            )

        with gr.Tab('2. 🖼️ Single-Image 4K Super-Resolution'):
            with gr.Row():
                with gr.Column(scale=3):
                    img_in = gr.Image(type='filepath', label='Input Portrait or Keyframe Image (.png / .jpg)', height=300)
                    with gr.Row():
                        img_res_sl = gr.Slider(720, 2160, value=1440, step=120, label='Target Short-Side Resolution (px)', info='1440p or 2160p (4K) restores crisp skin, eyes, and fabric detail on avatar portraits.')
                        img_max_sl = gr.Slider(0, 4096, value=3840, step=128, label='Max Long-Side Resolution (px)', info='3840 caps the longest edge at 4K UHD.')
                        img_tiled_cb = gr.Checkbox(value=True, label='Use Tiled VAE', info='Recommended for 1440p and 4K image upscaling.')
                    upscale_img_btn = gr.Button('🖼️ Upscale & Restore Image', variant='primary')
                with gr.Column(scale=2):
                    img_out = gr.Image(label='Restored High-Resolution PNG', height=360)
                    img_status_md = gr.Markdown('Ready.')
            upscale_img_btn.click(
                ui_upscale_image,
                inputs=[img_in, variant_dd, img_res_sl, img_max_sl, color_dd, blocks_swap_sl, img_tiled_cb, tile_sz_sl, seed_sl],
                outputs=[img_out, img_status_md],
            )

clear_output()
print('[UI] Launching SeedVR2 Gradio Studio ...')
demo.queue(max_size=4, concurrency_limit=1)
demo.load(
    lambda: f'✅ **SeedVR2 Ready** — Discovered **{len(DISCOVERED_VIDEOS)}** video(s) from your Drive notebooks.',
    outputs=[vid_status_md],
)
demo.launch(share=True, show_error=True, height=900, prevent_thread_lock=False)

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings Hugging Face API every 60 seconds in a background daemon thread so long 1080p/4K upscales do not disconnect.

import time, threading, datetime, requests, IPython
_stop_keepalive = threading.Event()

def _pinger():
    while not _stop_keepalive.is_set():
        try:
            requests.get('https://huggingface.co/api/models/ByteDance-Seed/SeedVR2-3B', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f"[Keep-Alive] {datetime.datetime.now().strftime('%H:%M:%S')} OK — stop with `_stop_keepalive.set()`.")
        except Exception as exc:
            print(f'[Keep-Alive] Warning: {exc}')
        _stop_keepalive.wait(60)

_ka_thread = threading.Thread(target=_pinger, daemon=True)
_ka_thread.start()
print('[Keep-Alive] Daemon started.')

In [ ]:
# @title STEP 6 — Quick Test (Headless Video Upscale + Audio Remux Verification)
# @markdown Upscales either the most recent cross-notebook Drive video (first 9 frames) or a generated
# @markdown low-res test clip with audio to verify end-to-end SeedVR2 restoration and `ffmpeg` audio preservation.

import os, time, subprocess, traceback
import cv2, numpy as np
from IPython.display import display, FileLink

QUICK_VARIANT = 'SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)'  # @param ['SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)', 'SeedVR2-3B FP16 (Standard 3B, L4 24GB / A100)', 'SeedVR2-7B Sharp FP8 Mixed (Extra Crisp 7B, L4 24GB / A100)']
QUICK_RESOLUTION = 720  # @param {type:"integer"}
QUICK_MAX_FRAMES = 9  # @param {type:"integer"}

try:
    # Prefer an existing sample from the repo or generate a synthetic 360p clip with audio
    sample_mp4 = pathlib.Path(SEEDVR2_REPO_DIR) / 'example_workflows' / 'example_inputs' / 'Mustache_640x360.mp4'
    test_in = str(sample_mp4) if sample_mp4.exists() else '/content/seedvr2_quicktest_in.mp4'
    if not os.path.exists(test_in):
        subprocess.run([
            'ffmpeg', '-y', '-f', 'lavfi', '-i', 'testsrc=duration=1:size=480x270:rate=9',
            '-f', 'lavfi', '-i', 'sine=frequency=440:duration=1',
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-shortest', test_in,
        ], check=True, capture_output=True)

    t0 = time.time()
    out_mp4 = ENGINE.upscale_media(
        input_path=test_in,
        variant_key=QUICK_VARIANT,
        resolution=QUICK_RESOLUTION,
        max_resolution=1280,
        batch_size=5,
        blocks_to_swap=16 if vram_gb < 20.0 else 0,
        load_cap=QUICK_MAX_FRAMES,
        preserve_audio=True,
    )
    print(f'[QuickTest] Completed in {time.time() - t0:.1f}s -> {out_mp4}')
    display(FileLink(str(out_mp4)))
except Exception as exc:
    traceback.print_exc()
    raise

In [ ]:
# @title STEP 7 — Batch Upscale Cross-Notebook Videos or Folder
# @markdown Batch-upscales all `.mp4` files in `BATCH_INPUT_DIR` (or the N most recent cross-notebook Drive outputs)
# @markdown with per-video `try`/`except` and automatic audio preservation.

import os, time, pathlib, traceback
from tqdm import tqdm
from IPython.display import display, FileLink

BATCH_VARIANT = 'SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)'  # @param ['SeedVR2-3B FP8 (Fastest, T4 16GB / L4 24GB)', 'SeedVR2-3B FP16 (Standard 3B, L4 24GB / A100)', 'SeedVR2-7B Sharp FP8 Mixed (Extra Crisp 7B, L4 24GB / A100)']
BATCH_INPUT_DIR = '/content/drive/MyDrive/AEI_3D_Out/InfiniteTalk'  # @param {type:"string"}
BATCH_RESOLUTION = 1080  # @param {type:"integer"}
BATCH_MAX_FILES = 5  # @param {type:"integer"}
BATCH_BLOCKS_SWAP = 12  # @param {type:"integer"}

in_dir = pathlib.Path(BATCH_INPUT_DIR)
if in_dir.exists():
    video_files = sorted(in_dir.glob('*.mp4'), key=lambda p: p.stat().st_mtime, reverse=True)
else:
    print(f'[Batch] {in_dir} not found; using discovered cross-notebook videos.')
    video_files = [pathlib.Path(p) for p in discover_cross_notebook_videos(max_results=BATCH_MAX_FILES).values()]

if BATCH_MAX_FILES > 0:
    video_files = video_files[:BATCH_MAX_FILES]

print(f'[Batch] Queued {len(video_files)} video(s) for {BATCH_RESOLUTION}p upscaling -> {DRIVE_OUT}')
ok_count, fail_count = 0, 0
for idx, vf in enumerate(tqdm(video_files, desc='SeedVR2 Batch'), 1):
    try:
        t0 = time.time()
        out_mp4 = ENGINE.upscale_media(
            input_path=str(vf),
            variant_key=BATCH_VARIANT,
            resolution=BATCH_RESOLUTION,
            max_resolution=1920 if BATCH_RESOLUTION <= 1080 else 3840,
            batch_size=5 if vram_gb < 20.0 else 9,
            blocks_to_swap=BATCH_BLOCKS_SWAP,
            chunk_size=45,
            preserve_audio=True,
        )
        print(f'  [{idx:02d}/{len(video_files)}] OK ({time.time() - t0:.1f}s) -> {out_mp4}')
        display(FileLink(str(out_mp4)))
        ok_count += 1
    except Exception as exc:
        fail_count += 1
        print(f'  [{idx:02d}/{len(video_files)}] FAILED on {vf.name}: {exc}')
        traceback.print_exc()

print(f'[Batch] Finished: {ok_count} succeeded, {fail_count} failed. Output folder: {DRIVE_OUT}')